# 1. Imports

In [40]:
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import f1_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
import pandas as pd
import numpy as np
import random

from jmetal.operator.mutation import PolynomialMutation
from jmetal.operator.crossover import SBXCrossover
from jmetal.algorithm.singleobjective import GeneticAlgorithm
from jmetal.operator.selection import BinaryTournamentSelection
from jmetal.core.solution import FloatSolution
from jmetal.util.termination_criterion import StoppingByEvaluations
from jmetal.core.problem import Problem

# 2. Decision Algos

In [41]:
# # Chargement datasets

# df_yeast = pd.read_csv("yeast.csv")
# print(df_yeast)
# X_yeast = df_yeast.drop("Output", axis=1).to_numpy()
# y_yeast = df_yeast["Output"].to_numpy()
# #run_xp_on_dataset('yeast1',X,y)

df_diabetes = pd.read_csv("diabetes.csv")
print(df_diabetes.columns)
X = df_diabetes.drop("Outcome", axis=1).to_numpy()
y = df_diabetes["Outcome"].to_numpy()
#run_xp_on_dataset('Pima',X,y)

Index(['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin',
       'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome'],
      dtype='object')


In [42]:
skf = StratifiedKFold(n_splits=3)
fold = 0
for train, test in skf.split(X, y):
    fold += 1
    for seed in range(3):
        X_train, X_test, y_train, y_test = X[train], X[test], y[train], y[test]

In [43]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

accuracy_rf = round(accuracy_score(y_test, y_pred_rf),3)
f1_rf = round(f1_score(y_test, y_pred_rf,average='binary'),3)
precision_rf = round(precision_score(y_test, y_pred_rf),3)
recall_rf = round(recall_score(y_test, y_pred_rf),3)

In [44]:
svc = SVC(kernel='rbf', random_state=42)
svc.fit(X_train, y_train)
y_pred_svc = svc.predict(X_test)

accuracy_svc = round(accuracy_score(y_test, y_pred_svc),3)
f1_svc = round(f1_score(y_test, y_pred_svc,average='binary'),3)
precision_svc = round(precision_score(y_test, y_pred_svc),3)
recall_svc = round(recall_score(y_test, y_pred_svc),3)

In [45]:
c45 = DecisionTreeClassifier(max_depth=4, random_state=42)
c45.fit(X_train, y_train)
y_pred_c45 = c45.predict(X_test)

accuracy_c45 = round(accuracy_score(y_test, y_pred_c45),3)
f1_c45 = round(f1_score(y_test, y_pred_c45,average='binary'),3)
precision_c45 = round(precision_score(y_test, y_pred_c45),3)
recall_c45 = round(recall_score(y_test, y_pred_c45),3)

In [46]:
print(f1_rf, f1_svc, f1_c45)

0.632 0.595 0.519


# 3. Implementation of Partial Classification

In [47]:
class PartialClassificationProblem(Problem):
    """
    Problème de classification partielle mono-objectif.
    Une solution = une règle avec bornes [bi, bs] pour chaque attribut.
    Objectif = maximiser F1 (donc minimiser -F1).
    """

    def __init__(self, X, y):
        # Appeler le constructeur parent en premier
        super().__init__()
        
        self.X = X
        self.y = y
        self.n_attributes = X.shape[1]

        # Définition des bornes globales des attributs
        mins = X.min(axis=0)
        maxs = X.max(axis=0)

        self.lower_bound = []
        self.upper_bound = []

        for i in range(self.n_attributes):
            self.lower_bound.extend([mins[i], mins[i]])
            self.upper_bound.extend([maxs[i], maxs[i]])

    def number_of_variables(self) -> int:
        return 2 * self.n_attributes

    def number_of_objectives(self) -> int:
        return 1

    def number_of_constraints(self) -> int:
        return 0

    def evaluate(self, solution: FloatSolution) -> FloatSolution:
        variables = solution.variables
        y_pred = []

        for instance in self.X:
            rule_active = True

            for i in range(self.n_attributes):
                bi = variables[2 * i]
                bs = variables[2 * i + 1]

                # Attribut actif si bi <= bs
                if bi <= bs:
                    if not (bi <= instance[i] <= bs):
                        rule_active = False
                        break

            y_pred.append(1 if rule_active else 0)

        # Gestion cas sans prédictions positives
        if sum(y_pred) == 0:
            f1 = 0.0
        else:
            f1 = f1_score(self.y, y_pred, average='binary', zero_division=0)

        solution.objectives[0] = -f1
        return solution

    def create_solution(self) -> FloatSolution:
        solution = FloatSolution(
            self.lower_bound,
            self.upper_bound,
            self.number_of_objectives(),
            self.number_of_constraints()
        )

        solution.variables = [
            random.uniform(self.lower_bound[i], self.upper_bound[i])
            for i in range(self.number_of_variables())
        ]

        return solution

    def name(self):
        return "PartialClassificationProblem"

In [48]:
#### Random Entries from Diabetes Dataset
diabetes_1 = df_diabetes[df_diabetes["Outcome"] == 1].sample(3)
diabetes_0 = df_diabetes[df_diabetes["Outcome"] == 0].sample(3)

In [51]:
# Création du problème avec vos données
problem = PartialClassificationProblem(X_train, y_train)

# Test de création et évaluation
solution = problem.create_solution()
evaluated_solution = problem.evaluate(solution)
print("F1 (négatif car minimisation) :", evaluated_solution.objectives[0])

# Configuration et lancement de l'algorithme génétique
algorithm = GeneticAlgorithm(
    problem=problem,
    population_size=100,
    offspring_population_size=100,
    mutation=PolynomialMutation(probability=1.0 / problem.number_of_variables()),
    crossover=SBXCrossover(probability=0.9),
    selection=BinaryTournamentSelection(),
    termination_criterion=StoppingByEvaluations(max_evaluations=25000)
)

algorithm.run()
result = algorithm.solutions[0]
# Affichage des résultats
print(f"\nMeilleure solution trouvée:")
print(f"F1 = {-result.objectives[0]:.3f}")

# Calcul des métriques détaillées
variables = result.variables
y_pred = []
for instance in X_train:
    rule_active = True
    for i in range(problem.n_attributes):
        bi = variables[2 * i]
        bs = variables[2 * i + 1]
        if bi <= bs:
            if not (bi <= instance[i] <= bs):
                rule_active = False
                break
    y_pred.append(1 if rule_active else 0)

precision = precision_score(y_train, y_pred, average='binary', zero_division=0)
recall = recall_score(y_train, y_pred, average='binary', zero_division=0)
f1 = f1_score(y_train, y_pred, average='binary', zero_division=0)

print(f"Precision = {precision:.3f}")
print(f"Recall = {recall:.3f}")
print(f"F1 = {f1:.3f}")

# Décodage de la règle
print("\nRègle décodée:")
for i in range(problem.n_attributes):
    bi = variables[2 * i]
    bs = variables[2 * i + 1]
    if bi <= bs:
        print(f"  Attribut {i}: [{bi:.2f}, {bs:.2f}]")
    else:
        print(f"  Attribut {i}: désactivé")

[2026-03-02 21:59:05,361] [jmetal.core.algorithm] [DEBUG] Creating initial set of solutions...
[2026-03-02 21:59:05,363] [jmetal.core.algorithm] [DEBUG] Evaluating solutions...
[2026-03-02 21:59:05,431] [jmetal.core.algorithm] [DEBUG] Initializing progress...
[2026-03-02 21:59:05,432] [jmetal.core.algorithm] [DEBUG] Running main loop until termination criteria is met


F1 (négatif car minimisation) : -0.0


[2026-03-02 21:59:32,136] [jmetal.core.algorithm] [DEBUG] Finished!



Meilleure solution trouvée:
F1 = 0.676
Precision = 0.589
Recall = 0.793
F1 = 0.676

Règle décodée:
  Attribut 0: désactivé
  Attribut 1: [99.98, 196.93]
  Attribut 2: désactivé
  Attribut 3: désactivé
  Attribut 4: désactivé
  Attribut 5: [26.32, 61.06]
  Attribut 6: désactivé
  Attribut 7: [24.70, 62.55]


# Evaluation et Analyse des Résultats